## 1. Seleccionar base de pruebas

In [1]:
import json

# Leer el archivo de preguntas y contextos esperados
with open('./testset/preguntas_contexto_esperado.json', encoding='utf-8') as f:
    preguntas_contexto = json.load(f)

# Extraer los tipos (por ejemplo: "fecha", "mes", "legislatura")
tipos_disponibles = list(preguntas_contexto.keys())

# Crear un diccionario con las preguntas por tipo
preguntas_por_tipo = {}
for tipo in tipos_disponibles:
    preguntas_por_tipo[tipo] = [item["query"] for item in preguntas_contexto[tipo]]

# Ahora tienes:
# - tipos_disponibles: lista de los tipos (['fecha', ...])
# - preguntas_por_tipo: diccionario {tipo: [pregunta1, pregunta2, ...]}

# Ejemplo de impresión para verificar
for tipo in tipos_disponibles:
    print(f"Tipo: {tipo} ({len(preguntas_por_tipo[tipo])} preguntas)")
    for pregunta in preguntas_por_tipo[tipo][:3]:  # Muestra solo las primeras 3 preguntas de cada tipo
        print("  -", pregunta)
    print()

Tipo: fecha (30 preguntas)
  - ¿Cuál fue la asistencia del Congreso el 22 de marzo de 2023?
  - ¿Cuál fue la asistencia del Congreso el 16 de diciembre de 2021?
  - ¿Cuál fue la asistencia del Congreso el 18 de septiembre de 2024?

Tipo: mes (10 preguntas)
  - Dame la asistencia del mes de julio del 2010
  - Dame la asistencia del mes de octubre del 2022
  - Dame la asistencia del mes de octubre del 2009

Tipo: legislatura (10 preguntas)
  - Dame los documentos de asistencia de la Primera Legislatura Ordinaria 2021-2022
  - Dame los documentos de asistencia de la Primera Legislatura Ordinaria 2010-2011
  - Dame los documentos de asistencia de la Legislatura extraordinaria 2010-2011



## 2. Recuperar con y generar contextos con ***qdrant***

In [2]:
import os
import json
import sys
import asyncio

# Asegura la ruta al módulo (relativa a esta carpeta: ../tools)
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "tools")))

from main_retriever import buscar_documentos_asistencia  # type: ignore

INPUT_FILE = './testset/preguntas_contexto_esperado.json'
OUTPUT_DIR = './testset'
OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'contexto_qdrant.json')

# Carga preguntas por tipo
with open(INPUT_FILE, encoding='utf-8') as f:
    preguntas_contexto = json.load(f)

tipos_disponibles = list(preguntas_contexto.keys())

async def construir_contexto_qdrant(modo: str = "all", sample_size: int = 2):
    """
    Construye y guarda el JSON en el formato deseado.
    - modo="sample": procesa solo `sample_size` preguntas por tipo.
    - modo="all": procesa todas las preguntas.
    """
    resultado_final = {tipo: [] for tipo in tipos_disponibles}

    for tipo in tipos_disponibles:
        items = preguntas_contexto[tipo]
        if modo == "sample":
            items = items[:sample_size]  # solo algunas preguntas

        for item in items:
            pregunta = item.get("query", "").strip()
            if not pregunta:
                continue

            try:
                res = await buscar_documentos_asistencia(pregunta)
            except Exception as e:
                res = {}

            docs = res.get("documentos", [])
            if isinstance(docs, str):
                context_list = [docs]
            elif isinstance(docs, list):
                context_list = [d for d in docs if isinstance(d, str) and d.strip()]
            else:
                context_list = []

            resultado_final[tipo].append({
                "query": pregunta,
                "context": context_list
            })

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(resultado_final, f, ensure_ascii=False, indent=2)

    print(f"✅ JSON guardado en: {OUTPUT_FILE}")


# Ejemplo de uso en notebook:
#await construir_contexto_qdrant(modo="sample", sample_size=2)
await construir_contexto_qdrant(modo="all")

🔁 Intento 1 de conexión a Qdrant...
✅ Conexión a Qdrant exitosa


HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/attendance_docs/points/query "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/attendance_docs/points/query "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/attendance_docs/points/query "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/attendance_docs/points/query "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/attendance_docs/points/query "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/att

✅ JSON guardado en: ./testset/contexto_qdrant.json
